# Sinus_CFD — nnU-Net training on NasalSeg (Google Colab)

Trains a 3D nnU-Net v2 segmentation model on the NasalSeg dataset (130 head CTs,
labels: background, left/right nasal cavity, nasopharynx, left/right maxillary
sinus). Goal: give the classical HU-threshold pipeline in `sinus_cfd.pipeline`
a learned alternative that can tell nasal-cavity air apart from sinus air
(see `docs/stage1_segmentation_baseline.md` for why that distinction is the
current bottleneck).

**Before running this notebook:**
1. On your machine: `py -3.12 scripts/download_nasalseg.py` then
   `py -3.12 scripts/prepare_nnunet_nasalseg.py --nasalseg-root data`
2. Zip `data/nnUNet_raw/Dataset501_NasalSeg/` and upload the zip to your
   Google Drive (e.g. `MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip`).
3. Runtime → Change runtime type → GPU (T4 is fine to start; A100 if you have
   Colab Pro/Pro+ and want it faster).

Colab sessions are ephemeral and can disconnect after ~12h (free tier: often
less, and idles out after ~90 min of inactivity). This notebook checkpoints
training results back to Drive after each run so a disconnect doesn't lose
progress — rerun the training cell with `--c` (continue) if it stops early.

## 1. Check GPU

In [1]:
!nvidia-smi

Fri Jul 17 20:42:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Install nnU-Net v2

In [3]:
!pip install -q nnunetv2>=2.5.0

## 3. Mount Drive and stage the dataset locally

nnU-Net does heavy small-file I/O during preprocessing/training; Drive-mounted
storage is much slower for that than Colab's local disk, so copy the zip in
and extract it to `/content` rather than pointing `nnUNet_raw` at Drive.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path if you uploaded the zip somewhere else in Drive
DRIVE_ZIP = '/content/drive/MyDrive/sinus_cfd/NasalSeg_nnUNet_raw.zip'
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/sinus_cfd/nnUNet_results'

Mounted at /content/drive


In [5]:
import os
os.makedirs('/content/nnUNet_data/nnUNet_raw', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_preprocessed', exist_ok=True)
os.makedirs('/content/nnUNet_data/nnUNet_results', exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

!unzip -q -o "$DRIVE_ZIP" -d /content/nnUNet_data/nnUNet_raw
!ls /content/nnUNet_data/nnUNet_raw

Dataset501_NasalSeg


## 4. Set nnU-Net environment variables

In [6]:
import os
os.environ['nnUNet_raw'] = '/content/nnUNet_data/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_data/nnUNet_preprocessed'
os.environ['nnUNet_results'] = '/content/nnUNet_data/nnUNet_results'
print(os.environ['nnUNet_raw'])
print(os.environ['nnUNet_preprocessed'])
print(os.environ['nnUNet_results'])

/content/nnUNet_data/nnUNet_raw
/content/nnUNet_data/nnUNet_preprocessed
/content/nnUNet_data/nnUNet_results


## 5. Plan and preprocess (dataset id 501 = NasalSeg)

This resamples all 130 cases to nnU-Net's chosen spacing and computes the
network configuration. Takes a few minutes for a dataset this size.

In [7]:
!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity

Fingerprint extraction...
Dataset501_NasalSeg
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Extracting dataset fingerprint: 100% 130/130 [00:05<00:00, 22.63it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [ 51.  186.5 152. ], 3d_lowres: [51, 186, 152]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_siz

## 6. Train fold 0 of 3d_fullres

Default is 1000 epochs. If the runtime disconnects partway through, rerun
this same cell with `--c` appended to resume from the last checkpoint
instead of restarting.

For an MVP, training **fold 0 only** (not the full 5-fold CV) is enough to
get a usable model and a held-out validation Dice; add folds 1-4 later if
you want a proper cross-validated estimate.

In [8]:
!nnUNetv2_train 501 3d_fullres 0 -tr nnUNetTrainer_250epochs


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-07-17 20:44:58.717293: Using torch.compile...
2026-07-17 20:45:01.800272: do_dummy_2d_data_aug: True
2026-07-17 20:45:01.801007: Creating new 5-fold cross-validation split...
2026-07-17 20:45:01.802902: Desired fold for training: 0
2026-07-17 20:45:01.803058: This split has 104 training and

## 7. Copy trained weights back to Drive

Colab's local disk does not persist across sessions — do this before the
runtime is recycled.

In [9]:
!rsync -av /content/nnUNet_data/nnUNet_results/ "$DRIVE_RESULTS_DIR/"
print('Synced to', DRIVE_RESULTS_DIR)

sending incremental file list
./
Dataset501_NasalSeg/
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/dataset.json
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/dataset_fingerprint.json
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/plans.json
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/checkpoint_best.pth
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/checkpoint_final.pth
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/debug.json
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/progress.png
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/training_log_2026_7_17_20_44_57.txt
Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_f

## 8. (Optional) Validation Dice summary

nnU-Net writes per-case validation metrics under
`nnUNet_results/Dataset501_NasalSeg/.../fold_0/validation/summary.json`.

In [10]:
import glob, json
summaries = glob.glob('/content/nnUNet_data/nnUNet_results/Dataset501_NasalSeg/**/summary.json', recursive=True)
print(summaries)
if summaries:
    s = json.load(open(summaries[0]))
    print(json.dumps(s.get('foreground_mean', s), indent=2))

['/content/nnUNet_data/nnUNet_results/Dataset501_NasalSeg/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_0/validation/summary.json']
{
  "Dice": 0.7897333427066189,
  "FN": 5761.638461538461,
  "FP": 5520.246153846153,
  "IoU": 0.7498698894254019,
  "TN": 1418966.2538461536,
  "TP": 19896.976923076923,
  "n_pred": 25417.223076923074,
  "n_ref": 25658.615384615383
}


## Next step back in the repo

Download `nnUNet_results/` from Drive to `data/nnUNet_results/` locally (or
just the best checkpoint), then run inference with:

```
nnUNetv2_predict -i INPUT_FOLDER -o OUTPUT_FOLDER -d 501 -c 3d_fullres -f 0
```

and compare its Dice against the same NasalSeg ground truth using
`scripts/evaluate_nasalseg_dice.py`'s methodology (per-case Dice vs labels
1-3), to see whether the learned model actually separates nasal-cavity air
from sinus air where the classical threshold pipeline couldn't.